# Validate Culture Grain

## Purpose

This notebook validates the transformation from the long-form ARMD
microbiology result table to the canonical culture fact table.

The raw and staging tables contain multiple rows per culture because
organisms, antibiotics, and susceptibility results are represented in
long form.

The final `fct_culture` model is expected to contain exactly one row per
culture order.

## Models evaluated

- `workspace.micro_raw.raw_armd_cohort`
- `workspace.micro_dev.stg_armd_cohort`
- `workspace.micro_dev.fct_culture`

## Validation questions

1. Does `fct_culture` contain one unique row per culture?
2. Are any order-level source attributes internally conflicting?
3. How many long-form source rows contribute to each culture?
4. Do raw, staging, and canonical row counts reconcile?

In [3]:
%sql
select
    count(*) as culture_rows,
    count(distinct culture_key) as distinct_culture_keys,
    sum(
        case
            when has_order_level_conflict then 1
            else 0
        end
    ) as culture_orders_with_conflicts
from workspace.micro_dev.fct_culture

,culture_rows,distinct_culture_keys,culture_orders_with_conflicts
0,751075,751075,0


## Culture-grain validation

Expected results for ARMD release `2025-04-11`:

| Measure | Expected |
|---|---:|
| Culture rows | 751,075 |
| Distinct culture keys | 751,075 |
| Orders with order-level conflicts | 0 |

Equal row and key counts confirm the intended one-row-per-culture grain.

Zero order-level conflicts means the long-form rows belonging to each
culture agree on the culture-level attributes used in the canonical
model.

In [4]:
%sql
select
    source_row_count,
    count(*) as culture_orders
from workspace.micro_dev.fct_culture
group by source_row_count
order by source_row_count

,source_row_count,culture_orders
0,1,632612
1,2,456
2,3,703
3,4,3257
4,5,6289
5,6,2641
6,7,6447
7,8,7331
8,9,7613
9,10,2488


## Long-form source-row distribution

`source_row_count` measures how many records from the long-form
microbiology source contribute to each canonical culture order.

Cultures with susceptibility testing may contribute many source rows,
while cultures without multiple organism-antibiotic results may
contribute relatively few.

This distribution demonstrates why the 2.24 million source records
cannot be interpreted as 2.24 million cultures.

In [5]:
%sql
select
    'raw_armd_cohort' as model_name,
    count(*) as row_count
from workspace.micro_raw.raw_armd_cohort

union all

select
    'stg_armd_cohort' as model_name,
    count(*) as row_count
from workspace.micro_dev.stg_armd_cohort

union all

select
    'fct_culture' as model_name,
    count(*) as row_count
from workspace.micro_dev.fct_culture

,model_name,row_count
0,raw_armd_cohort,2241050
1,stg_armd_cohort,2241050
2,fct_culture,751075


## Pipeline row-count reconciliation

Expected results:

| Model | Expected rows |
|---|---:|
| `raw_armd_cohort` | 2,241,050 |
| `stg_armd_cohort` | 2,241,050 |
| `fct_culture` | 751,075 |

The raw and staging row counts should match because staging standardizes
fields without changing the source grain.

The reduction from 2,241,050 staged rows to 751,075 fact rows reflects
the intentional transformation from long-form microbiology results to
one canonical row per culture order.

## Conclusion

The culture model is considered validated when:

- `culture_rows` equals `distinct_culture_keys`;
- no order-level conflicts are present;
- raw and staging row counts match;
- the canonical fact contains the expected 751,075 culture orders.

This validated culture grain is the foundation for the downstream
culture-organism, susceptibility, cohort, and reporting models.